[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-text_features/examples/text_features/text_features.ipynb)

# Use the answers as features

A downstream model sometimes wants a number per text, not a chat completion. FRIDA-Decisions turns each review into a few calibrated-looking features: three yes/no probabilities and one score. A logistic regression, fit in code on a toy split, then decides which reviews to escalate. Swapping a question changes a column of the matrix; it does not change the regression.

This is a dozen labelled reviews, not a benchmark. The printout includes a majority-class baseline so the regression has something dull to beat.

Pattern: a small version of [Autoresearch feature discovery](https://docs.typesafe.ai/cookbooks/autoresearch_feature_discovery). Labels are synthetic.


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0" scikit-learn


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
try:
    from sklearn.linear_model import LogisticRegression
except ImportError as error:
    raise SystemExit("pip install scikit-learn") from error

FEATURES = ("angry", "demand", "concrete", "severity")
SEVERITY = [
    "Спокойный отзыв, делать ничего не нужно.",
    "Лёгкое недовольство, можно ответить шаблоном.",
    "Есть претензия, стоит прочитать.",
    "Явная претензия, лучше ответить человеку.",
    "Жёсткая претензия, откладывать нельзя.",
]


def request_for(text: str) -> dict:
    return {
        "state": text,
        "questions": {
            "angry": {"type": "noul", "instructions": "Автор раздражён?"},
            "demand": {"type": "noul", "instructions": "Автор требует вернуть деньги или что-то исправить?"},
            "concrete": {"type": "noul", "instructions": "Назван конкретный заказ, товар или дата?"},
            "severity": {"type": "score", "instructions": "Насколько отзыв требует реакции?",
                         "criteria": SEVERITY},
        },
    }


def featurize(answers: dict) -> list[float]:
    return [
        answers["angry"]["noul"],
        answers["demand"]["noul"],
        answers["concrete"]["noul"],
        answers["severity"]["score"] / 4.0,
    ]


## The regression only sees the four numbers

The split is fixed in `data.json`. Accuracy on eight test reviews is a sanity check. The coefficients show which question the linear model leaned on.


In [4]:
def run(judge, rows: list[dict]) -> None:
    responses = judge.judge_batch([request_for(row["text"]) for row in rows])
    matrix = [featurize(response["answers"]) for response in responses]
    train = [(x, row["escalate"]) for x, row in zip(matrix, rows) if row["split"] == "train"]
    test = [(x, row["escalate"]) for x, row in zip(matrix, rows) if row["split"] == "test"]
    model = LogisticRegression(max_iter=400)
    model.fit([x for x, _ in train], [y for _, y in train])
    predicted = model.predict([x for x, _ in test])
    gold = [y for _, y in test]
    hits = sum(int(left == right) for left, right in zip(predicted, gold))
    majority = max(set(y for _, y in train), key=lambda label: sum(y == label for _, y in train))
    baseline = sum(int(label == majority) for label in gold)
    print("features:", ", ".join(FEATURES))
    print("coefficients:", " ".join(f"{name}={coef:.2f}" for name, coef in zip(FEATURES, model.coef_[0])))
    print(f"test accuracy {hits}/{len(test)}")
    print(f"majority baseline (label {majority}) {baseline}/{len(test)}")
    for row, pred in zip((row for row in rows if row["split"] == "test"), predicted):
        mark = "ok" if pred == row["escalate"] else "miss"
        print(f"  {mark:4} pred {int(pred)} gold {row['escalate']}  {row['text'][:64]}")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-text-features/examples/text_features/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


features: angry, demand, concrete, severity
coefficients: angry=0.62 demand=0.89 concrete=-0.36 severity=0.99
test accuracy 7/7
majority baseline (label 1) 3/7
  ok   pred 1 gold 1  Заказ 3308 протекает, верните оплату, я не буду это чинить сам.
  ok   pred 1 gold 1  Обещали доставку 4 апреля и не привезли. Исправьте или верните д
  ok   pred 0 gold 0  Лампа яркая, мужу понравилась, претензий нет.
  ok   pred 0 gold 0  Обычный кабель, работает, как и ждал.
  ok   pred 1 gold 1  Пылесос дымит с первого включения, заказ 6620, жду замену и изви
  ok   pred 0 gold 0  Понравился сервис, ответили быстро и по делу.
  ok   pred 0 gold 0  Чашка целая, цвет как на фото.
